# 3차 파인튜닝 (v3) — 층화 배합으로 v2의 근접 퇴화 수정

**v2 실패 원인:** 실전 크롭이 저해상 위주 + 8배 오버샘플 → 학습이 저해상으로 쏠려 근접 큰 글자 퇴화 (600: -1, 700: -3).

**v3 처방:** 크롭을 근접(close)/저해상(low)/페어(pair)로 태그해 **배합 비율을 명시 제어**:
기본 = 합성×1 + 근접×6 + 저해상×2 + 페어×6. 평가(val)도 근접/저해상 따로 보고 → 어느 쪽이 늘고 줄었는지 바로 확인.

**업로드:** `synth_rec.zip` + `real_rec_data_v3.zip` · GPU(T4) 런타임 · 셀1 후 세션 재시작

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 데이터 업로드(2개) + 층화 배합
MIX = {'close': 6, 'low': 2, 'pair': 6}   # ← 배합 비율 (여기만 바꿔서 재실험 가능)

from google.colab import files
up = files.upload()   # synth_rec.zip, real_rec_data_v3.zip 둘 다 선택
!unzip -oq synth_rec.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data_v3.zip -d PaddleOCR/train_data/
import io
def read(p): return [l.split('\t') for l in io.open(p, encoding='utf-8').read().splitlines()]
synth_tr = [f'synth_rec/train/{p}\t{t}' for p, t in read('PaddleOCR/train_data/synth_rec/train/rec_gt_train.txt')]
merged = list(synth_tr)
from collections import Counter; used = Counter()
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_train.txt'):
    merged += [f'real_rec_data_v3/{p}\t{t}'] * MIX.get(g, 1); used[g] += MIX.get(g, 1)
va_close, va_low = [], []
for p, t, g in read('PaddleOCR/train_data/real_rec_data_v3/meta_val.txt'):
    (va_close if g == 'close' else va_low).append(f'real_rec_data_v3/{p}\t{t}')
io.open('PaddleOCR/train_data/train_v3.txt', 'w', encoding='utf-8').write('\n'.join(merged) + '\n')
io.open('PaddleOCR/train_data/val_close.txt', 'w', encoding='utf-8').write('\n'.join(va_close) + '\n')
io.open('PaddleOCR/train_data/val_low.txt', 'w', encoding='utf-8').write('\n'.join(va_low) + '\n')
io.open('PaddleOCR/train_data/val_all.txt', 'w', encoding='utf-8').write('\n'.join(va_close + va_low) + '\n')
print(f'train {len(merged)} (합성 {len(synth_tr)} + 실전배합 {sum(used.values())} {dict(used)})')
print(f'val 근접 {len(va_close)} · 저해상 {len(va_low)}')

In [ ]:
# 3) config·사전학습 모델 자동 탐색
%cd /content
import glob, os
cfgs = glob.glob('PaddleOCR/configs/rec/**/*korean*', recursive=True)
CFG = next((c for c in cfgs if 'v5' in c.lower() and 'mobile' in c.lower()), cfgs[0] if cfgs else None)
print('사용 config:', CFG)
urls = [
 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
 'https://paddleocr.bj.bcebos.com/PP-OCRv5/multilingual/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
]
for u in urls:
    if os.system(f'wget -q {u} -O pretrain.pdparams') == 0 and os.path.getsize('pretrain.pdparams') > 1e6:
        print('사전학습 확보:', u); break

In [ ]:
# 4) 학습 (20 epochs, T4 ~60-80분) — 베스트 선택은 근접+저해상 합산 val 기준
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/train.py -c {CFG_REL} \
  -o Global.pretrained_model=/content/pretrain \
     Global.epoch_num=20 \
     Global.save_model_dir=./output/korean_lowres_v3 \
     Global.eval_batch_step="[0,500]" \
     Optimizer.lr.learning_rate=0.0001 \
     Train.sampler.first_bs=32 \
     Train.dataset.data_dir=./train_data \
     Train.dataset.label_file_list=["./train_data/train_v3.txt"] \
     Eval.dataset.data_dir=./train_data \
     Eval.dataset.label_file_list=["./train_data/val_all.txt"]

In [ ]:
# 5) 근접/저해상 분리 평가 — v2 대비 근접 회복 여부를 여기서 판정
%cd /content/PaddleOCR
for name in ['val_close', 'val_low']:
    print('=====', name, '=====')
    !python tools/eval.py -c {CFG_REL} \
      -o Global.pretrained_model=./output/korean_lowres_v3/best_accuracy \
         Eval.dataset.data_dir=./train_data \
         Eval.dataset.label_file_list=["./train_data/{name}.txt"] 2>/dev/null | grep -E 'acc|norm'

In [ ]:
# 6) 추론 모델로 내보내기 + 다운로드
%cd /content/PaddleOCR
!python tools/export_model.py -c {CFG_REL} \
  -o Global.pretrained_model=./output/korean_lowres_v3/best_accuracy \
     Global.save_inference_dir=./korean_lowres_v3_rec_infer
!zip -q -r /content/korean_lowres_v3_rec_infer.zip korean_lowres_v3_rec_infer
from google.colab import files
files.download('/content/korean_lowres_v3_rec_infer.zip')
print('로컬 A/B: daelim_closeup.py --rec_dir korean_lowres_v3_rec_infer (캐시 자동 분리: v3)')